# Part 1, extension 3a check (version 3): search the whole rectangle with a working Peru border layer

When notebook 07 searched for the 0.25° window with the most forest loss, Python warned "invalid value encountered in subtract". The warning means some cells of the 1 km search grid had no valid number. A window containing such a cell gets no score, so it could never be picked, even if it had more loss than the window that was chosen.

This notebook rereads the same 1 km search grid (one small Earth Engine request, no images) and checks three things:

1. **Where the missing cells are**, and whether any are inside Peru.
2. **The best window among windows with no missing cells**, which should be the one notebook 07 used.
3. **Whether any window with missing cells could beat it.** Every missing cell is counted as 100% lost, the most it could possibly be. If no window wins even then, the choice is confirmed.

**Version 3.** Version 2 showed the forest-loss layer is complete, but 11% of the Peru border layer came back as "infinite", all east of longitude −70.44. That layer was made by clipping a constant image to Peru's outline. Here it is drawn a different way, by painting Peru's outline onto an empty image. Section 1 stops the notebook if any cell of the new layer is broken, and compares it with the old layer wherever the old one worked.

**Version 2.** The first version turned infinite values into huge numbers instead of treating them as missing, so a few windows could be wrongly ruled out as "not inside Peru". This version treats every non-finite value (empty, +infinity or −infinity) as missing in both layers. When a cell's Peru value is missing, the strict test counts the cell as outside Peru, and the "could it beat ours" test counts it as inside.

It changes nothing. If the window turns out not to be the busiest, it says so, and deciding what to do about it comes next.

In [1]:
EE_PROJECT = "earth-01-final"
SEARCH_BOX = [-71.5, -13.5, -68.7, -11.0]   # as in notebook 07
WINDOW_DEG = 0.25
COARSE_DEG = 0.01
TREECOVER_MIN = 30
GRID_JSON = "../data/cleaned/part1_map_grid.json"

import json
import numpy as np
import ee

ee.Authenticate()
ee.Initialize(project=EE_PROJECT, opt_url="https://earthengine-highvolume.googleapis.com")
ee.data.setDeadline(180_000)

USED = json.load(open(GRID_JSON))["window"]      # the window notebook 07 actually downloaded
print("Window used by notebooks 07 and 08:", [float(v) for v in USED])

/opt/anaconda3/lib/python3.13/site-packages/google/api_core/_python_package_support.py:206: FutureWarning: Package google.api_core depends on grpcio, currently installed at version 1.75.1. grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, Google Cloud Python packages will raise their minimum requirements (including google-api-core, grpcio, and grpcio-status) to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(


Window used by notebooks 07 and 08: [-70.69, -12.89, -70.44, -12.64]


In [2]:
# Same search grid as notebook 07, section 3 (code copied unchanged)
hansen = ee.Image("UMD/hansen/global_forest_change_2023_v1_11")
treecover = hansen.select("treecover2000").unmask(0)
lossyear = hansen.select("lossyear").unmask(0)
datamask = hansen.select("datamask").unmask(0)
forest2000 = treecover.gte(TREECOVER_MIN).And(datamask.eq(1))
peru = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq("ADM0_NAME", "Peru")).geometry()
recent = forest2000.And(lossyear.gte(19)).And(lossyear.lte(23))
peru_fc = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq("ADM0_NAME", "Peru"))
search = (recent.toFloat().reduceResolution(ee.Reducer.mean(), maxPixels=4096).rename("recent")
          .addBands(ee.Image(0).paint(peru_fc, 1).toFloat().rename("peru"))                    # new way
          .addBands(ee.Image.constant(1).clip(peru).unmask(0).toFloat().rename("peru_old")))    # notebook 07's way

w, s, e, n = SEARCH_BOX
W, H = int(round((e - w) / COARSE_DEG)), int(round((n - s) / COARSE_DEG))
arr = ee.data.computePixels({
    "expression": search, "fileFormat": "NUMPY_NDARRAY",
    "grid": {"dimensions": {"width": W, "height": H},
             "affineTransform": {"scaleX": COARSE_DEG, "shearX": 0, "translateX": w,
                                 "shearY": 0, "scaleY": -COARSE_DEG, "translateY": n},
             "crsCode": "EPSG:4326"}})
recent_map, peru_map = arr["recent"].astype(float), arr["peru"].astype(float)
peru_old = arr["peru_old"].astype(float)

/opt/anaconda3/lib/python3.13/site-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for UMD/hansen/global_forest_change_2023_v1_11! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by UMD/hansen/global_forest_change_2025_v1_13

Learn more: https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2023_v1_11

  warnings.warn(warning, category=DeprecationWarning)


## 1. Where are the missing cells?

In [3]:
bad_loss = ~np.isfinite(recent_map)
bad_peru = ~np.isfinite(peru_map)
bad = bad_loss | bad_peru
print(f"Missing cells: {bad.sum()} of {bad.size} ({bad.mean():.2%}); "
      f"in the loss layer {bad_loss.sum()}, in the Peru layer {bad_peru.sum()}")
print(f"  Kinds: empty {np.isnan(recent_map).sum() + np.isnan(peru_map).sum()}, "
      f"infinite {np.isinf(recent_map).sum() + np.isinf(peru_map).sum()}")
if bad.any():
    rows, cols = np.where(bad)
    print(f"  They span lon {w + cols.min() * COARSE_DEG:.2f} to {w + (cols.max() + 1) * COARSE_DEG:.2f}, "
          f"lat {n - (rows.max() + 1) * COARSE_DEG:.2f} to {n - rows.min() * COARSE_DEG:.2f}")
    print("  ", "(none expected in version 3)")
    in_peru = bad_loss & ~bad_peru & (np.where(bad_peru, 0.0, peru_map) >= 0.5)
    print(f"  Cells with missing loss that are known to be inside Peru: {in_peru.sum()}")
assert not bad.any(), "the new layers still have broken cells; stop here and send this output to Claude"

ok_old = np.isfinite(peru_old)
agree = (peru_map[ok_old] >= 0.5) == (peru_old[ok_old] >= 0.5)
print(f"Old Peru layer: {(~ok_old).sum()} broken cells. Where it worked, the new layer agrees on "
      f"{agree.mean():.2%} of {ok_old.sum()} cells")
print(f"Cells inside Peru (new layer): {(peru_map >= 0.5).mean():.1%} of the rectangle")
assert agree.mean() > 0.99, "the new Peru layer disagrees with the old one; stop here"

Missing cells: 0 of 70000 (0.00%); in the loss layer 0, in the Peru layer 0
  Kinds: empty 0, infinite 0
Old Peru layer: 7685 broken cells. Where it worked, the new layer agrees on 99.68% of 62315 cells
Cells inside Peru (new layer): 88.7% of the rectangle


## 2 and 3. Best window, with missing cells left out and with missing cells counted as fully lost

In [4]:
def window_sums(a, k):     # as in notebook 07
    c = np.pad(a, ((1, 0), (1, 0))).cumsum(0).cumsum(1)
    return c[k:, k:] - c[:-k, k:] - c[k:, :-k] + c[:-k, :-k]

k = int(round(WINDOW_DEG / COARSE_DEG))
peru_strict = np.where(bad_peru, 0.0, peru_map)                # missing counts as outside Peru
peru_maybe = np.where(bad_peru, 1.0, peru_map)                 # missing counts as inside Peru
inside = window_sums(peru_strict, k) >= k**2 - 0.5             # surely fully inside Peru
inside_maybe = window_sums(peru_maybe, k) >= k**2 - 0.5        # possibly fully inside Peru
assert np.isfinite(window_sums(peru_strict, k)).all() and np.isfinite(window_sums(peru_maybe, k)).all()
has_bad = window_sums(bad.astype(float), k) > 0.5              # contains a missing cell

share_clean = window_sums(np.where(bad, 0.0, recent_map), k) / k**2
share_upper = window_sums(np.where(bad, 1.0, recent_map), k) / k**2   # missing cells counted as 100% lost

def to_window(i, j):
    return [float(round(w + j * COARSE_DEG, 4)), float(round(n - (i + k) * COARSE_DEG, 4)),
            float(round(w + (j + k) * COARSE_DEG, 4)), float(round(n - i * COARSE_DEG, 4))]

clean = inside & ~has_bad
assert clean.any(), "no window inside Peru is free of missing cells"
i, j = np.unravel_index(np.argmax(np.where(clean, share_clean, -1)), share_clean.shape)
best_clean = to_window(i, j)
print(f"Busiest window with no missing cells: {best_clean}, loss {share_clean[i, j]:.2%}")

i_used = int(round((n - USED[3]) / COARSE_DEG)); j_used = int(round((USED[0] - w) / COARSE_DEG))
used_share = share_clean[i_used, j_used]
print(f"Window used by notebooks 07 and 08:   {[float(v) for v in USED]}, loss {used_share:.2%}"
      f"{'  (contains missing cells)' if has_bad[i_used, j_used] else ''}")

assert np.isfinite(share_clean).all() and np.isfinite(share_upper).all()
rivals = inside_maybe & (has_bad | ~inside) & (share_upper > used_share)
print(f"Windows with missing cells that could beat it, even counting every missing cell as fully lost: {rivals.sum()}")

same = np.allclose(best_clean, USED, atol=1e-6)
if same and not rivals.any():
    print("\nConfirmed: the window used is the busiest 0.25° window inside Peru in the search rectangle.")
elif same:
    print("\nNot fully confirmed: the window used is the busiest of the clean windows, but some windows with "
          "missing cells could be busier. Send this output to Claude before changing anything.")
else:
    print("\nNot confirmed: a different window has more loss. Send this output to Claude before changing anything.")

Busiest window with no missing cells: [-70.06, -12.9, -69.81, -12.65], loss 16.05%
Window used by notebooks 07 and 08:   [-70.69, -12.89, -70.44, -12.64], loss 7.94%
Windows with missing cells that could beat it, even counting every missing cell as fully lost: 0

Not confirmed: a different window has more loss. Send this output to Claude before changing anything.
